In [1]:
import pandas as pd
import pm4py
import duckdb

print("pandas:", pd.__version__)
print("pm4py:", pm4py.__version__)
print("duckdb:", duckdb.__version__)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




pandas: 3.0.0
pm4py: 2.7.23.11
duckdb: 1.5.6


In [2]:
import time

start = time.time()
df = pm4py.read_xes("data/BPI Challenge 2017.xes.gz")
print("Seconds taken:", round(time.time() - start, 1))

print("Type:", type(df))
print("Rows, columns:", df.shape)
print(df.columns.tolist())

C:\Users\RAJA VINAY KUMAR\AppData\Roaming\Python\Python312\site-packages\pm4py\utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

Seconds taken: 58.7
Type: <class 'pandas.DataFrame'>
Rows, columns: (1202267, 19)
['Action', 'org:resource', 'concept:name', 'EventOrigin', 'EventID', 'lifecycle:transition', 'time:timestamp', 'case:LoanGoal', 'case:ApplicationType', 'case:concept:name', 'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms', 'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount', 'OfferID']


In [3]:
df.to_parquet("data/bpi2017.parquet", index=False)

check = pd.read_parquet("data/bpi2017.parquet")
print("Saved rows:", len(check))
print("Same as original:", len(check) == len(df))

Saved rows: 1202267
Same as original: True


In [4]:
info = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "example": df.iloc[0],
    "missing_%": (df.isna().mean() * 100).round(1)
})
print(info.to_string())

                                      type                           example  missing_%
Action                                 str                           Created        0.0
org:resource                           str                            User_1        0.0
concept:name                           str              A_Create Application        0.0
EventOrigin                            str                       Application        0.0
EventID                                str             Application_652823628        0.0
lifecycle:transition                   str                          complete        0.0
time:timestamp         datetime64[us, UTC]  2016-01-01 09:51:15.304000+00:00        0.0
case:LoanGoal                          str            Existing loan takeover        0.0
case:ApplicationType                   str                        New credit        0.0
case:concept:name                      str             Application_652823628        0.0
case:RequestedAmount            

In [5]:
print(df["EventOrigin"].value_counts())
print()
print(df[df["OfferedAmount"].notna()]["concept:name"].value_counts())

EventOrigin
Workflow       768823
Application    239595
Offer          193849
Name: count, dtype: int64

concept:name
O_Create Offer    42995
Name: count, dtype: int64


In [6]:
steps = df["concept:name"].value_counts()
print("Number of different steps:", len(steps))
print(steps.to_string())

Number of different steps: 26
concept:name
W_Validate application        209496
W_Call after offers           191092
W_Call incomplete files       168529
W_Complete application        148900
W_Handle leads                 47264
O_Create Offer                 42995
O_Created                      42995
O_Sent (mail and online)       39707
A_Validating                   38816
A_Create Application           31509
A_Concept                      31509
A_Accepted                     31509
A_Complete                     31362
O_Returned                     23305
A_Incomplete                   23055
O_Cancelled                    20898
A_Submitted                    20423
O_Accepted                     17228
A_Pending                      17228
A_Cancelled                    10431
O_Refused                       4695
A_Denied                        3753
W_Assess potential fraud        3282
O_Sent (online only)            2026
W_Shortened completion           238
W_Personal Loan collection      

In [7]:
endings = ["A_Pending", "A_Cancelled", "A_Denied"]
end_df = df[df["concept:name"].isin(endings)]

per_app = end_df.groupby("case:concept:name")["concept:name"].nunique()
print("Apps with 1 ending:", (per_app == 1).sum())
print("Apps with 2+ endings:", (per_app > 1).sum())
print("Apps with no ending:", df["case:concept:name"].nunique() - len(per_app))

Apps with 1 ending: 31411
Apps with 2+ endings: 0
Apps with no ending: 98


In [8]:
ts = df["time:timestamp"]
print("First event:", ts.min())
print("Last event:", ts.max())

first_per_app = df.groupby("case:concept:name")["time:timestamp"].min()
print("Last application created:", first_per_app.max())

First event: 2016-01-01 09:51:15.304000+00:00
Last event: 2017-02-01 14:11:03.499000+00:00
Last application created: 2016-12-31 21:37:53.216000+00:00
